In [ ]:
"""
NOTEBOOK: DEEP LEARNING EXPERIMENTS
====================================
Purpose: Experiment with neural networks for complex patterns
Output: PyTorch models for production
"""

# 🧠 Deep Learning Experiments Notebook

**Objective:** Explore neural network architectures for price prediction

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

class LSTMPredictor(nn.Module):
    """
    LSTM for time series price prediction
    Exports to: backend/app/ml/deep/lstm_price.py
    """
    
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        # LSTM forward
        out, _ = self.lstm(x, (h0, c0))
        
        # Take last time step
        out = out[:, -1, :]
        
        # Fully connected layer
        out = self.fc(out)
        
        return out

class DeepPricePredictor:
    """
    Deep Learning wrapper for production
    Exports to: backend/app/ml/deep/price_engine.py
    """
    
    def __init__(self, sequence_length=30):
        self.sequence_length = sequence_length
        self.model = None
        self.optimizer = None
        self.criterion = nn.MSELoss()
        self.train_losses = []
        self.val_losses = []
    
    def prepare_sequences(self, prices):
        """Convert price series to sequences for LSTM"""
        X, y = [], []
        
        for i in range(len(prices) - self.sequence_length):
            X.append(prices[i:i+self.sequence_length])
            y.append(prices[i+self.sequence_length])
        
        X = np.array(X).reshape(-1, self.sequence_length, 1)
        y = np.array(y).reshape(-1, 1)
        
        return torch.FloatTensor(X), torch.FloatTensor(y)
    
    def train(self, prices, epochs=100, batch_size=32, learning_rate=0.001):
        """Train LSTM model"""
        print("🧠 Training LSTM model...")
        
        # Prepare data
        X, y = self.prepare_sequences(prices)
        
        # Split train/val
        split = int(0.8 * len(X))
        X_train, X_val = X[:split], X[split:]
        y_train, y_val = y[:split], y[split:]
        
        # Create data loaders
        train_dataset = TensorDataset(X_train, y_train)
        val_dataset = TensorDataset(X_val, y_val)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        
        # Initialize model
        self.model = LSTMPredictor(input_size=1, hidden_size=64, num_layers=2).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        # Training loop
        for epoch in range(epochs):
            # Training
            self.model.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                self.optimizer.zero_grad()
                predictions = self.model(batch_X)
                loss = self.criterion(predictions, batch_y)
                loss.backward()
                self.optimizer.step()
                
                train_loss += loss.item()
            
            # Validation
            self.model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    predictions = self.model(batch_X)
                    loss = self.criterion(predictions, batch_y)
                    val_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            self.train_losses.append(avg_train_loss)
            self.val_losses.append(avg_val_loss)
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
        print("✅ LSTM training complete!")
        
        return self.model
    
    def predict(self, sequence):
        """Predict next price from sequence"""
        self.model.eval()
        with torch.no_grad():
            input_tensor = torch.FloatTensor(sequence).reshape(1, -1, 1).to(device)
            prediction = self.model(input_tensor)
            return prediction.cpu().numpy()[0, 0]
    
    def save_model(self, version="v1"):
        """Save PyTorch model"""
        import os
        os.makedirs("../../backend/ml_weights/deep", exist_ok=True)
        
        path = f"../../backend/ml_weights/deep/lstm_price_{version}.pt"
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'train_losses': self.train_losses,
            'val_losses': self.val_losses
        }, path)
        
        print(f"💾 Deep learning model saved to: {path}")
        return path

print("\n✅ Deep learning experiments loaded!")